# 07 – Kueri Parafrase (jenis opsional pada Tabel 2)

Masukan: `queries.parquet` (notebook 05). Keluaran: `queries_para.parquet` (kueri valid + kolom `q_para`),
`paraphrase_review.csv` (60 sampel untuk diperiksa manual), `paraphrase_summary.json`.

Fragmen asli (`q_exact`) ditulis ulang oleh model bahasa instruksi (Qwen2.5) dengan kata berbeda tetapi makna sama.
Parafrase dianggap valid bila: panjangnya wajar (0,5–2× kata asli, minimal 4 kata), tidak identik dengan asli, dan
tumpang tindih kata isi dengan fragmen asli ≤ `MAX_OVERLAP`. Percobaan pertama greedy; yang gagal dicoba ulang
dengan sampling (suhu 0,7 lalu 0,9). Yang tetap gagal dibuang.

Karena dibuat dari fragmen yang sama, hasil evaluasinya berpasangan dengan exact, parsial, dan berisik.

In [1]:
# Konfigurasi dan data
import os, sys, re, glob, json, time
import numpy as np, pandas as pd
from tqdm.auto import tqdm

sys.path.insert(0, os.path.dirname(glob.glob("/kaggle/input/datasets/nshamid/lyrecall-songs-clean/lyrecall_text.py", recursive=True)[0]))
from lyrecall_text import normalize_lex

def find(name):
    return glob.glob(f"/kaggle/input/**/{name}", recursive=True)[0]

# ====== KONFIGURASI ======
MODEL_ID    = "Qwen/Qwen2.5-3B-Instruct"      # alternatif: "Qwen/Qwen2.5-1.5B-Instruct"
BATCH       = 32
MAX_NEW     = 96
MAX_OVERLAP = 0.5       # maks porsi kata isi asli yang boleh muncul lagi di parafrase
N_LIMIT     = None        
SEED        = 42
OUT_DIR     = "/kaggle/working"

Q = pd.read_parquet(find("queries.parquet")).reset_index(drop=True)
Qs = Q.sample(N_LIMIT, random_state=SEED).reset_index(drop=True) if N_LIMIT else Q.copy()
print("Kueri yang diparafrase:", len(Qs), Qs["split"].value_counts().to_dict())

Kueri yang diparafrase: 4000 {'test': 3000, 'val': 1000}


In [2]:
# Muat model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(SEED)
assert torch.cuda.is_available(), "Aktifkan Accelerator GPU"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to("cuda")
model.eval()
print("GPU:", torch.cuda.get_device_name(0), "| memori terpakai (GB):",
      round(torch.cuda.memory_allocated() / 1e9, 1))

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

GPU: Tesla T4 | memori terpakai (GB): 6.3


In [3]:
# Prompt dan fungsi generasi
FEWSHOT = [
    ("i keep driving through the empty streets all night",
     "through deserted roads i cruise until the morning light"),
    ("you broke my heart and left me standing in the cold",
     "my chest still aches from the way you walked off and abandoned me outside"),
]
SYSTEM = "You rewrite short song lyric excerpts."
INSTR = ("Rewrite the lyric excerpt below so it keeps the same meaning but uses completely different words and phrasing. "
         "Do not copy phrases from the original. Keep it about the same length. "
         "Output only the rewritten text on a single line, with no quotes and no explanation.\n\nExcerpt: ")

def build_prompt(frag):
    msgs = [{"role": "system", "content": SYSTEM}]
    for a, b in FEWSHOT:
        msgs += [{"role": "user", "content": INSTR + a}, {"role": "assistant", "content": b}]
    msgs.append({"role": "user", "content": INSTR + frag})
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def gen_llm(frags, temp):
    enc = tok([build_prompt(f) for f in frags], return_tensors="pt", padding=True).to("cuda")
    kw = dict(max_new_tokens=MAX_NEW, pad_token_id=tok.eos_token_id)
    kw.update(dict(do_sample=True, temperature=temp, top_p=0.9) if temp > 0 else dict(do_sample=False))
    out = model.generate(**enc, **kw)
    return tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

# uji satu batch kecil
for f, p in zip(Qs["q_exact"][:3], gen_llm(Qs["q_exact"][:3].tolist(), 0.0)):
    print("ASLI :", f, "\nHASIL:", p, "\n")

ASLI : big wave do do do big wave do do do big wave its gonna hit you with a big wave 
HASIL: mighty cresting swell swells swells mighty cresting swell it's coming for you with a mighty swell 

ASLI : dont wanna skip that go on a trip yeah go on a trip yeah you let it slip no dont wanna skip that 
HASIL: don't wish to miss this chance for a journey yes, let's embark on a trip yes, you missed out no, don't want to pass up this opportunity 

ASLI : erything they said we couldnt ahh jokes on you you on twitter lyin bout what you do while you watch me do it on youtube 
HASIL: every promise they made seemed fake you lied on socials while I monitored your online actions 



In [4]:
# Pembersihan, validasi dan generasi massal
STOP = set("a an the and or but if of to in on at by for with from as is am are was were be been it its i me my you your we us our he she they them his her their this that these those not no so do does did dont im ill ive youre cant wont just up out all".split())

def clean_out(txt):
    lines = [l.strip() for l in txt.strip().split("\n") if l.strip()]
    if not lines:
        return ""
    s = re.sub(r"^(rewritten( text| excerpt)?|rewrite|paraphrase|output)\s*:\s*", "", lines[0], flags=re.I)
    return s.strip(" \"'“”")

def check(orig, para):
    """Parafrase valid: cukup panjang, tidak sama, dan tidak banyak memakai kata isi asli."""
    ow, pw = orig.split(), para.split()
    if len(pw) < 4 or not (0.5 * len(ow) <= len(pw) <= 2.0 * len(ow)) or para == orig:
        return False, 1.0
    uniq = set(ow)
    content = {w for w in uniq if w not in STOP} or uniq
    ov = len(content & set(pw)) / len(content)
    return ov <= MAX_OVERLAP, ov

def generate_all(frags, gen_fn):
    n = len(frags)
    out, raw, att, ov = [None] * n, [""] * n, [0] * n, [np.nan] * n
    pending = list(range(n))
    for attempt, temp in enumerate([0.0, 0.7, 0.9], 1):
        if not pending:
            break
        order = sorted(pending, key=lambda i: len(frags[i]))      # panjang mirip -> batch efisien
        nxt = []
        for b in tqdm(range(0, len(order), BATCH), desc=f"percobaan {attempt} (suhu {temp})"):
            idx = order[b:b + BATCH]
            for i, t in zip(idx, gen_fn([frags[i] for i in idx], temp)):
                raw[i] = clean_out(t)
                p = normalize_lex(raw[i])
                ok, o = check(frags[i], p)
                if ok:
                    out[i], att[i], ov[i] = p, attempt, o
                else:
                    nxt.append(i)
        pending = nxt
        print(f"  setelah percobaan {attempt}: belum valid {len(pending)} dari {n}")
    return out, raw, att, ov

t0 = time.time()
out, raw, att, ov = generate_all(Qs["q_exact"].tolist(), gen_llm)
print(f"Selesai: {time.time() - t0:.0f} dtk")

percobaan 1 (suhu 0.0):   0%|          | 0/125 [00:00<?, ?it/s]

  setelah percobaan 1: belum valid 118 dari 4000


percobaan 2 (suhu 0.7):   0%|          | 0/4 [00:00<?, ?it/s]

  setelah percobaan 2: belum valid 50 dari 4000


percobaan 3 (suhu 0.9):   0%|          | 0/2 [00:00<?, ?it/s]

  setelah percobaan 3: belum valid 24 dari 4000
Selesai: 718 dtk


In [5]:
# Ringkasan dan contoh untuk diperiksa
Qp = Qs.copy()
Qp["q_para"], Qp["para_raw"], Qp["para_attempt"], Qp["para_overlap"] = out, raw, att, ov
n_all = len(Qp)
Qp = Qp[Qp["q_para"].notna()].reset_index(drop=True)
print(f"Parafrase valid: {len(Qp)} dari {n_all} ({len(Qp) / n_all:.1%})")
print("Per split:", Qp["split"].value_counts().to_dict())
print("Percobaan ke-:", Qp["para_attempt"].value_counts().sort_index().to_dict())

def overlap_with(orig, other):
    uniq = set(orig.split()); content = {w for w in uniq if w not in STOP} or uniq
    return len(content & set(other.split())) / len(content)
print("\nTumpang tindih kata isi dengan kueri asli (rerata):")
for col in ["q_partial", "q_noisy", "q_para"]:
    print(f"  {col:10s}: {np.mean([overlap_with(a, b) for a, b in zip(Qp.q_exact, Qp[col])]):.2f}")

print("\nContoh (periksa maknanya):")
for _, r in Qp.sample(min(20, len(Qp)), random_state=SEED).iterrows():
    print(f"[{r.qid}] {r.q_exact}\n      -> {r.q_para}")

Parafrase valid: 3976 dari 4000 (99.4%)
Per split: {'test': 2981, 'val': 995}
Percobaan ke-: {1: 3882, 2: 68, 3: 26}

Tumpang tindih kata isi dengan kueri asli (rerata):
  q_partial : 0.73
  q_noisy   : 0.78
  q_para    : 0.15

Contoh (periksa maknanya):
[val-00410] lord we lift our hands in worship
      -> god we raise our palms in praise
[test-00430] stop crying shut your mouth give it up let it go
      -> dry your tears silence your plea release the hold let it be
[val-00685] i can fix your wildest needs i got heaven and i got hell roll on up for my price is down
      -> for any desire you might have ive got both extremes at my disposal just pay the cost my rates are quite low
[test-02174] and if you want to go walkin dear its delightful its delicious its de lovely
      -> and if you wish to stroll sweetie its charming its scrumptious its delightful
[test-01369] yuh tryna play imma bust your brain for real doe yeah i can fuck i can shoot uh huh i can do both mh hm
      -> you w

In [6]:
# Simpan
Qp.to_parquet(f"{OUT_DIR}/queries_para.parquet", index=False)
rev = Qp.sample(min(60, len(Qp)), random_state=SEED)[["qid", "q_exact", "q_para"]].copy()
rev["makna_sama_1_0"] = ""
rev.to_csv(f"{OUT_DIR}/paraphrase_review.csv", index=False)
with open(f"{OUT_DIR}/paraphrase_summary.json", "w") as f:
    json.dump({"model": MODEL_ID, "n_input": int(n_all), "n_valid": int(len(Qp)),
               "success_rate": round(len(Qp) / n_all, 4), "max_overlap": MAX_OVERLAP,
               "mean_overlap": round(float(Qp.para_overlap.mean()), 3),
               "attempts": {int(k): int(v) for k, v in Qp.para_attempt.value_counts().items()},
               "seed": SEED}, f, indent=2)
for fn in sorted(os.listdir(OUT_DIR)):
    print(f"{fn:28s} {os.path.getsize(os.path.join(OUT_DIR, fn)) / 1e6:8.2f} MB")

__notebook__.ipynb               0.03 MB
paraphrase_review.csv            0.01 MB
paraphrase_summary.json          0.00 MB
queries_para.parquet             1.73 MB
